## Run model and collect predictions on BABILong

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import time
from pathlib import Path
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
import openai
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
# openai

base_url = os.environ['LLM_BASE_URL']
# Load API key from environment
token = os.environ["LLM_API_KEY"]
# token loaded from environment variable LLM_API_KEY
client = openai.OpenAI(api_key=token, base_url=base_url)
results_folder = './babilong_evals_2/'
model_name = 'llama-3.3-70b-instruct'

In [ ]:
# TODO: Update path to your local BABILong data directory
dataset_name = './babilong_data'

tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
split_names = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k', '128k'] #['32k'] ['64k', '128k']

use_instruction = True
use_examples = True
use_post_prompt = True
use_chat_template = True

In [ ]:
# define generation parameters
generate_kwargs = {
    'max_tokens': 20,
    'n': 1,
    'temperature': 0.0,
}
# test request
messages =  [
                {
                    "role": "system",
                    "content": "You are a intelligent assistant."
                },
                {
                    "role": "user",
                    "content": 'what can you do?',
                },
            ]
response = client.chat.completions.create(model=model_name, messages=messages, **generate_kwargs)
response.choices[0].message.content

In [ ]:
def process_sample(sample):
    target = sample['target']
    context = sample['input']
    question = sample['question']

    # format input text
    input_text = get_formatted_input(context, question, prompt_cfg['examples'],
                                        prompt_cfg['instruction'], prompt_cfg['post_prompt'],
                                        template=prompt_cfg['template'])
    messages = [
        {
            "role": "system",
            "content": "You are a intelligent assistant."
        },
        {
            "role": "user",
            "content": input_text,
        },
    ]

    try:
        response = client.chat.completions.create(model=model_name, messages=messages, **generate_kwargs)
        print(response)
    except openai.InternalServerError as e:
        print(e)
        response = None
    return response

def process_sample_with_retry(sample, iterations=0):
    try:
        return process_sample(sample)
    except openai.RateLimitError as e:
        if iterations >= 3:
            return None
        else:
            time.sleep(15)
            return process_sample_with_retry(sample, iterations+1)

In [ ]:
from babilong.prompts import DEFAULT_TEMPLATE, DEFAULT_PROMPTS, get_formatted_input

print(f'prompt template:\n{DEFAULT_TEMPLATE}')

for task in tqdm(tasks, desc='tasks'):
    # configure the prompt
    prompt_cfg = {
        'instruction': DEFAULT_PROMPTS[task]['instruction'] if use_instruction else '',
        'examples': DEFAULT_PROMPTS[task]['examples'] if use_examples else '',
        'post_prompt': DEFAULT_PROMPTS[task]['post_prompt'] if use_post_prompt else '',
        'template': DEFAULT_TEMPLATE,
        'chat_template': use_chat_template,
    }
    prompt_name = [f'{k}_yes' if prompt_cfg[k] else f'{k}_no' for k in prompt_cfg if k != 'template']
    prompt_name = '_'.join(prompt_name)

    for split_name in tqdm(split_names, desc='lengths'):
        # load dataset
        print("Loading dataset", task, split_name)
        data_path = f"{dataset_name}/{task}/{split_name}.json"
        task_data = json.load(open(data_path))
        #data = datasets.load_dataset(dataset_name, split_name)
        #task_data = data[task]
        
        # Prepare files with predictions, prompt, and generation configurations
        outfile = Path(f'{results_folder}/{model_name}/{task}_{split_name}_{prompt_name}.csv')
        print(outfile)
        if not os.path.exists(outfile):
            outfile.parent.mkdir(parents=True, exist_ok=True)
            cfg_file = f'{results_folder}/{model_name}/{task}_{split_name}_{prompt_name}.json'
            json.dump({'prompt': prompt_cfg, 'generate_kwargs': generate_kwargs}, open(cfg_file, 'w'), indent=4)

            df = pd.DataFrame({'target': [], 'output': [], 'question': []})

            for sample in tqdm(task_data, desc=f'task: {task} length: {split_name}'):
                target = sample['target']
                question = sample['question']
                response = process_sample_with_retry(sample)
                if response is not None and response.choices is not None:
                    output = response.choices[0].message.content.strip()
                else:
                    output = ''

                df.loc[len(df)] = [target, output, question]
                # write results to csv file
                df.to_csv(outfile)
        else:
            print(f"Skipping {outfile}. Already present!")

### Get scores

In [ ]:
from babilong.metrics import compare_answers, TASK_LABELS
import matplotlib.pylab as plt
import matplotlib
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

import numpy as np

tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k', '128k']

In [ ]:
accuracy = np.ones((len(tasks), len(lengths))) * -1
for j, task in enumerate(tasks):
    for i, ctx_length in enumerate(lengths):
        prompt_cfg = {
            'instruction': DEFAULT_PROMPTS[task]['instruction'] if use_instruction else '',
            'examples': DEFAULT_PROMPTS[task]['examples'] if use_examples else '',
            'post_prompt': DEFAULT_PROMPTS[task]['post_prompt'] if use_post_prompt else '',
            'template': DEFAULT_TEMPLATE,
            'chat_template': use_chat_template,
        }
        prompt_name = [f'{k}_yes' if prompt_cfg[k] else f'{k}_no' for k in prompt_cfg if k != 'template']
        prompt_name = '_'.join(prompt_name)

        fname = f'{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
        if not os.path.isfile(fname):
            continue
        
        df = pd.read_csv(fname)
        
        if df['output'].dtype != object:
            df['output'] = df['output'].astype(str)
        df['output'] = df['output'].fillna('')


        df['correct'] = df.apply(lambda row: compare_answers(target=row['target'], output=row['output'],
                                                             question=row['question'], task_labels=TASK_LABELS[task]),
                                                             axis=1)
        score = df['correct'].sum()
        accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0

# Set large font sizes for better visibility in the PDF
matplotlib.rc('font', size=14)
# Create a colormap for the heatmap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
figsize=(5, 3.5)
fig, ax = plt.subplots(1, 1, figsize=figsize)  # Adjust the size as necessary

sns.heatmap(accuracy, cmap=cmap, vmin=0, vmax=100, annot=True, fmt=".0f",
            linewidths=.5, xticklabels=lengths, yticklabels=tasks, ax=ax)

prompt_name = '+'.join([k for k in ['instruction', 'examples'] if prompt_cfg[k] != ''])
ax.set_title(f'{prompt_name}\n{model_name}')
ax.set_xlabel('Context size')
ax.set_ylabel('Tasks')
plt.show()

In [ ]:
results = {}
model_names = ['llama-3.3-70b-instruct']
for model_name in model_names:
    prompt_name = 'instruction_no_examples_no_post_prompt_no'
    accuracy = np.ones((len(tasks), len(lengths))) * -1
    try:
        for j, task in enumerate(tasks):
            for i, ctx_length in enumerate(lengths):
                fname = f'{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    # print(f'No such file: {fname}')
                    continue
                
                df = pd.read_csv(fname)
                
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')


                df['correct'] = df.apply(lambda row: compare_answers(target=row['target'], output=row['output'], question=row['question'],
                                                                     task_labels=TASK_LABELS[task]), axis=1)
                score = df['correct'].sum()
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0

        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes'
        accuracy_fs = np.ones((len(tasks), len(lengths))) * -1
        for j, task in enumerate(tasks):
            for i, ctx_length in enumerate(lengths):
                fname = f'{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    # print(f'No such file: {fname}')
                    continue
                
                df = pd.read_csv(fname)
                
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')


                df['correct'] = df.apply(lambda row: compare_answers(target=row['target'], output=row['output'], question=row['question'],
                                                                     task_labels=TASK_LABELS[task]), axis=1)
                score = df['correct'].sum()
                accuracy_fs[j, i] = 100 * score / len(df) if len(df) > 0 else 0

        results[model_name] = {'zero': accuracy, 'few': accuracy_fs}
        

        # Set large font sizes for better visibility in the PDF
        matplotlib.rc('font', size=14)

        # Create a colormap for the heatmap
        cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)

        # Create the heatmap
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))  # Adjust the size as necessary
        
        sns.heatmap(accuracy, cmap=cmap, vmin=0, vmax=100, annot=True, fmt=".0f",
                    linewidths=.5, xticklabels=lengths, yticklabels=tasks, ax=ax[0])
        
        sns.heatmap(accuracy_fs, cmap=cmap, vmin=0, vmax=100, annot=True, fmt=".0f",
                    linewidths=.5, xticklabels=lengths, yticklabels=tasks, ax=ax[1])
        
        ax[0].set_title(f'Zero shot \n {model_name}')
        ax[1].set_title(f'Few shot \n {model_name}')
        ax[0].set_xlabel('Context size')
        ax[1].set_xlabel('Context size')
        ax[0].set_ylabel('Tasks')
        ax[1].set_ylabel('Tasks')

        # Save the figure to a PDF
        # plt.savefig('all_tasks_performance.pdf', bbox_inches='tight')
        plt.show()
    except KeyError as e:
        print(f"question not found in {model_name}")
        continue
        